# 02b_cold_start_embeddings.ipynb
**Cold-Start Embeddings via k-NN + Developer Reputation**

## Estrategia
Para cada juego post-2016 (sin embedding entrenado), encontramos los k juegos pre-2016
más similares por contenido (Steam metadata + RAWG features) y promediamos sus embeddings RS.
Además calculamos un feature de **reputación del desarrollador** basado en el historial pre-2016.

## Outputs
- `../Data/item_embeddings_rs_coldstart.npy` — shape (3682, 64), pre-2016: reales | post-2016: k-NN
- `../Data/developer_reputation.npy`          — shape (3682,), reputación por juego

In [1]:
import pandas as pd
import numpy as np
import json, ast, warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix

INTERACTIONS  = "../Data/interactions.parquet"
ITEMMAP       = "../Data/item2idx.json"
ITEM_EMB      = "../Data/item_embeddings_rs_clean.npy"
STEAM_GAMES   = "../Data/steam_games.json"
RAWG_CSV      = "../Data/rawg_enriched.csv"
OUTPUT_EMB    = "../Data/item_embeddings_rs_coldstart.npy"
OUTPUT_DEVREP = "../Data/developer_reputation.npy"
CUTOFF        = pd.Timestamp('2016-01-01')
K_NEIGHBORS   = 10

item_emb = np.load(ITEM_EMB)
with open(ITEMMAP, "r") as f:
    item2idx = {k: int(v) for k, v in json.load(f).items()}

df_inter   = pd.read_parquet(INTERACTIONS)
target_df  = df_inter.groupby("item_idx").size().reset_index(name="total_reviews")

print(f"Embeddings: {item_emb.shape}")
print(f"Items:      {len(item2idx)}")
print(f"Reviews por juego: {len(target_df)} juegos con interacciones")

Embeddings: (3682, 64)
Items:      3682
Reviews por juego: 3682 juegos con interacciones


## Carga y fusión de datos Steam + RAWG

In [2]:
def safe_len(x):
    try: return len(x) if isinstance(x, list) else 0
    except: return 0

def clean_price(p):
    if pd.isna(p) or p in ('Free', ''): return 0.0
    if isinstance(p, (int, float)): return float(p)
    try: return float(str(p).replace('$','').replace('\u20ac','').replace(',','').strip())
    except: return 0.0

def list_to_text(x):
    if isinstance(x, list): return ' '.join(str(i).replace(' ','_') for i in x)
    return ''

games = []
with open(STEAM_GAMES, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: games.append(ast.literal_eval(line))
        except: pass

df_steam = pd.json_normalize(games).rename(columns={"id": "item_id"})
df_steam["item_idx"] = df_steam["item_id"].map(item2idx)
df_steam = df_steam.dropna(subset=["item_idx"])
df_steam["item_idx"] = df_steam["item_idx"].astype(int)
df_steam['release_date_parsed'] = pd.to_datetime(df_steam['release_date'], errors='coerce')
df_steam['price']         = df_steam['price'].apply(clean_price)
df_steam['early_access']  = df_steam['early_access'].fillna(0).astype(int)
df_steam['num_genres']    = df_steam['genres'].apply(safe_len)
df_steam['num_tags']      = df_steam['tags'].apply(safe_len)
df_steam['num_specs']     = df_steam['specs'].apply(safe_len)
df_steam['has_sentiment'] = df_steam['sentiment'].notna().astype(int)
df_steam['steam_tags_text']   = df_steam['tags'].apply(list_to_text)
df_steam['steam_genres_text'] = df_steam['genres'].apply(list_to_text)

print(f"Steam: {len(df_steam)} juegos, {df_steam['release_date_parsed'].notna().sum()} con fecha")

Steam: 3195 juegos, 3107 con fecha


In [3]:
df_rawg = pd.read_csv(RAWG_CSV)

# Features numericos RAWG
df_rawg['rawg_rating']       = pd.to_numeric(df_rawg['rawg_rating'], errors='coerce').fillna(0)
df_rawg['playtime_avg_h']    = pd.to_numeric(df_rawg['playtime_avg_h'], errors='coerce').fillna(0)
df_rawg['rawg_ratings_count']= pd.to_numeric(df_rawg['rawg_ratings_count'], errors='coerce').fillna(0)
df_rawg['metacritic']        = pd.to_numeric(df_rawg['metacritic'], errors='coerce').fillna(0)
df_rawg['has_metacritic']    = (df_rawg['metacritic'] > 0).astype(int)

# Contar plataformas
df_rawg['num_platforms'] = df_rawg['platforms_str'].fillna('').apply(
    lambda x: len([p for p in x.split('|') if p.strip()]) if x else 0
)

# Texto RAWG tags (mas ricos que Steam)
df_rawg['rawg_tags_text']    = df_rawg['tags_str'].fillna('').apply(
    lambda x: ' '.join(t.strip().replace(' ','_') for t in x.split('|') if t.strip())
)
df_rawg['rawg_genres_text']  = df_rawg['genres_str'].fillna('').apply(
    lambda x: ' '.join(g.strip().replace(' ','_') for g in x.split('|') if g.strip())
)

RAWG_NUM_COLS = ['rawg_rating', 'playtime_avg_h', 'rawg_ratings_count',
                 'metacritic', 'has_metacritic', 'num_platforms']

print(f"RAWG: {len(df_rawg)} juegos")
print(f"Cobertura numerica: {df_rawg[RAWG_NUM_COLS].gt(0).mean().round(2).to_dict()}")

RAWG: 3195 juegos
Cobertura numerica: {'rawg_rating': 0.67, 'playtime_avg_h': 0.79, 'rawg_ratings_count': 0.78, 'metacritic': 0.36, 'has_metacritic': 0.36, 'num_platforms': 0.82}


In [4]:
# Fusionar Steam + RAWG + target
df = df_steam.merge(
    df_rawg[['item_idx'] + RAWG_NUM_COLS + ['rawg_tags_text','rawg_genres_text','developers_str','publishers_str']],
    on='item_idx', how='left'
).merge(
    target_df, on='item_idx', how='left'
)
df['total_reviews'] = df['total_reviews'].fillna(0).astype(int)
for c in RAWG_NUM_COLS: df[c] = df[c].fillna(0)
for c in ['rawg_tags_text','rawg_genres_text','developers_str','publishers_str']:
    df[c] = df[c].fillna('')

# Filtrar a juegos con fecha valida y embedding disponible
df_valid = df[
    df['release_date_parsed'].notna() &
    (df['item_idx'] < len(item_emb))
].copy().reset_index(drop=True)

print(f"Dataset fusionado: {len(df_valid)} juegos con fecha y embedding")
print(f"  Con datos RAWG:  {(df_valid['rawg_rating'] > 0).sum()} juegos")
print(f"  Con metacritic:  {(df_valid['metacritic'] > 0).sum()} juegos")

Dataset fusionado: 3107 juegos con fecha y embedding
  Con datos RAWG:  2077 juegos
  Con metacritic:  1135 juegos


## Developer Reputation
Para cada juego calculamos el promedio de reviews de los juegos pre-2016
del mismo desarrollador. Es un proxy del historial de calidad/popularidad del estudio.

In [5]:
# Parsear lista de developers
def parse_devs(s):
    if not s or pd.isna(s): return []
    return [d.strip() for d in str(s).split('|') if d.strip()]

df_valid['dev_list'] = df_valid['developers_str'].apply(parse_devs)

# Construir mapa developer → media de reviews (solo pre-2016)
pre2016_df = df_valid[df_valid['release_date_parsed'] < CUTOFF]
dev_reviews_map = {}
for _, row in pre2016_df.iterrows():
    for dev in row['dev_list']:
        dev_reviews_map.setdefault(dev, []).append(row['total_reviews'])

dev_rep_score = {dev: np.mean(vals) for dev, vals in dev_reviews_map.items()}

print(f"Developers con historial pre-2016: {len(dev_rep_score)}")
top5 = sorted(dev_rep_score.items(), key=lambda x: -x[1])[:5]
print("Top 5 por reputacion:")
for dev, score in top5:
    print(f"  {dev:30s}: {score:.1f} reviews promedio")

# Aplicar a todos los juegos
def compute_rep(dev_list):
    scores = [dev_rep_score[d] for d in dev_list if d in dev_rep_score]
    return np.mean(scores) if scores else 0.0

df_valid['developer_reputation'] = df_valid['dev_list'].apply(compute_rep)

pre_mask  = (df_valid['release_date_parsed'] < CUTOFF).values
post_mask = ~pre_mask
print(f"\nReputacion en pre-2016:  {df_valid[pre_mask]['developer_reputation'].describe().round(2).to_dict()}")
print(f"Reputacion en post-2016: {df_valid[post_mask]['developer_reputation'].describe().round(2).to_dict()}")

Developers con historial pre-2016: 2012
Top 5 por reputacion:
  Hidden Path Entertainment     : 1881.0 reviews promedio
  Facepunch Studios             : 1275.0 reviews promedio
  Engine Software               : 741.0 reviews promedio
  Pipeworks Studio              : 741.0 reviews promedio
  Re-Logic                      : 741.0 reviews promedio



Reputacion en pre-2016:  {'count': 2621.0, 'mean': 15.79, 'std': 68.9, 'min': 0.0, '25%': 1.0, '50%': 2.0, '75%': 8.5, 'max': 1881.0}
Reputacion en post-2016: {'count': 486.0, 'mean': 7.75, 'std': 34.26, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 1.0, 'max': 391.17}


In [6]:
# Guardar developer_reputation como array indexado por item_idx (shape: n_items)
dev_rep_array = np.zeros(len(item_emb), dtype=np.float32)
for _, row in df_valid.iterrows():
    dev_rep_array[row['item_idx']] = row['developer_reputation']

np.save(OUTPUT_DEVREP, dev_rep_array)
print(f"Guardado: {OUTPUT_DEVREP}")
print(f"  Shape: {dev_rep_array.shape}")
print(f"  Juegos con reputacion > 0: {(dev_rep_array > 0).sum()}")
print(f"  Media (juegos con info):   {dev_rep_array[dev_rep_array > 0].mean():.2f}")

Guardado: ../Data/developer_reputation.npy
  Shape: (3682,)
  Juegos con reputacion > 0: 2217
  Media (juegos con info):   20.36


## Construccion de features enriquecidos para k-NN

In [7]:
STEAM_NUM = ['price','early_access','num_genres','num_tags','num_specs','has_sentiment']

# TF-IDF: Steam tags
tfidf_steam_tags = TfidfVectorizer(max_features=150, min_df=2)
mat_steam_tags   = tfidf_steam_tags.fit_transform(df_valid['steam_tags_text'].fillna(''))

# TF-IDF: Steam genres
tfidf_steam_gen  = TfidfVectorizer(max_features=30, min_df=2)
mat_steam_gen    = tfidf_steam_gen.fit_transform(df_valid['steam_genres_text'].fillna(''))

# TF-IDF: RAWG tags (mas ricos)
tfidf_rawg_tags  = TfidfVectorizer(max_features=150, min_df=2)
mat_rawg_tags    = tfidf_rawg_tags.fit_transform(df_valid['rawg_tags_text'].fillna(''))

# TF-IDF: RAWG genres
tfidf_rawg_gen   = TfidfVectorizer(max_features=30, min_df=2)
mat_rawg_gen     = tfidf_rawg_gen.fit_transform(df_valid['rawg_genres_text'].fillna(''))

# Numericas: Steam + RAWG
num_steam = csr_matrix(df_valid[STEAM_NUM].fillna(0).values)
num_rawg  = csr_matrix(df_valid[RAWG_NUM_COLS].fillna(0).values)

# Combinar todo
X_all = hstack([num_steam, num_rawg, mat_steam_tags, mat_steam_gen,
                mat_rawg_tags, mat_rawg_gen]).toarray()

print(f"Feature matrix enriquecida: {X_all.shape}")
print(f"  Steam numeric:  {num_steam.shape[1]}")
print(f"  RAWG numeric:   {num_rawg.shape[1]}")
print(f"  Steam tags:     {mat_steam_tags.shape[1]}")
print(f"  Steam genres:   {mat_steam_gen.shape[1]}")
print(f"  RAWG tags:      {mat_rawg_tags.shape[1]}")
print(f"  RAWG genres:    {mat_rawg_gen.shape[1]}")

Feature matrix enriquecida: (3107, 354)
  Steam numeric:  6
  RAWG numeric:   6
  Steam tags:     150
  Steam genres:   23
  RAWG tags:      150
  RAWG genres:    19


## k-NN Cold-Start
Para cada juego post-2016, encontramos los k=10 juegos pre-2016 mas similares
por similitud coseno en el espacio de features enriquecido.
El embedding inferido es el promedio ponderado de sus embeddings RS reales.

In [8]:
pre2016_mask  = (df_valid['release_date_parsed'] < CUTOFF).values
post2016_mask = ~pre2016_mask

idx_pre  = df_valid[pre2016_mask]['item_idx'].values
idx_post = df_valid[post2016_mask]['item_idx'].values

# Normalizar para similitud coseno
X_pre_norm  = normalize(X_all[pre2016_mask])
X_post_norm = normalize(X_all[post2016_mask])
y_pre       = item_emb[idx_pre]

print(f"Pre-2016:  {len(idx_pre)} juegos (k-NN training)")
print(f"Post-2016: {len(idx_post)} juegos (a inferir)")

Pre-2016:  2621 juegos (k-NN training)
Post-2016: 486 juegos (a inferir)


In [9]:
# Ajustar k-NN sobre el espacio pre-2016
knn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric='cosine', algorithm='brute', n_jobs=-1)
knn.fit(X_pre_norm)

# Buscar vecinos para post-2016
distances_post, indices_post = knn.kneighbors(X_post_norm)
similarities_post = 1 - distances_post   # cosine similarity (1 = identico)

# Inferir embeddings: promedio ponderado (softmax de similitudes)
def weighted_avg_emb(sims, idxs, embeddings):
    sims = np.clip(sims, 0, None)          # similitudes negativas → 0
    w = np.exp(sims) / np.exp(sims).sum()  # softmax
    return (w[:, np.newaxis] * embeddings[idxs]).sum(axis=0)

y_post_knn = np.stack([
    weighted_avg_emb(similarities_post[i], indices_post[i], y_pre)
    for i in range(len(idx_post))
])

print(f"Embeddings inferidos shape: {y_post_knn.shape}")
print(f"Similitud coseno media con vecinos mas cercanos: {similarities_post[:, 0].mean():.4f}")
print(f"Similitud coseno media con k-esimo vecino:       {similarities_post[:, -1].mean():.4f}")

Embeddings inferidos shape: (486, 64)
Similitud coseno media con vecinos mas cercanos: 0.9865
Similitud coseno media con k-esimo vecino:       0.9747


## Evaluacion de calidad
Evaluamos el k-NN con cross-validation dentro de los juegos pre-2016:
dejamos un fold fuera, usamos el resto como pool de vecinos, e inferimos.

In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2_folds   = []
cv_cosim_folds = []

print(f"Cross-validation k-NN (k={K_NEIGHBORS}, 5 folds, solo pre-2016):")
for fold, (tr, te) in enumerate(kf.split(X_pre_norm), 1):
    knn_cv = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric='cosine', algorithm='brute')
    knn_cv.fit(X_pre_norm[tr])
    dists, idxs = knn_cv.kneighbors(X_pre_norm[te])
    sims = 1 - dists

    y_hat = np.stack([
        weighted_avg_emb(sims[i], idxs[i], y_pre[tr])
        for i in range(len(te))
    ])
    y_true = y_pre[te]

    r2s   = [r2_score(y_true[:, d], y_hat[:, d]) for d in range(64)]
    cosims = cosine_similarity(y_true, y_hat).diagonal()

    cv_r2_folds.append(np.mean(r2s))
    cv_cosim_folds.append(cosims.mean())
    print(f"  Fold {fold}: R2={np.mean(r2s):.4f}  cosine_sim={cosims.mean():.4f}")

print(f"\nR2 CV:         {np.mean(cv_r2_folds):.4f} +/- {np.std(cv_r2_folds):.4f}")
print(f"Cosine sim CV: {np.mean(cv_cosim_folds):.4f} +/- {np.std(cv_cosim_folds):.4f}")
print()
if np.mean(cv_r2_folds) > 0.30:   print("-> Inferencia util")
elif np.mean(cv_r2_folds) > 0.10: print("-> Inferencia parcial")
else:                              print("-> Inferencia debil")

Cross-validation k-NN (k=10, 5 folds, solo pre-2016):


  Fold 1: R2=-0.0990  cosine_sim=0.0654


  Fold 2: R2=-0.0753  cosine_sim=0.0666


  Fold 3: R2=-0.0808  cosine_sim=0.0600
  Fold 4: R2=-0.0760  cosine_sim=0.0646


  Fold 5: R2=-0.0946  cosine_sim=0.0622

R2 CV:         -0.0851 +/- 0.0098
Cosine sim CV: 0.0637 +/- 0.0024

-> Inferencia debil


## Combinar y guardar embeddings cold-start

In [11]:
# Construir array final: pre-2016 reales, post-2016 k-NN inferidos
item_emb_cs = item_emb.copy()
item_emb_cs[idx_post] = y_post_knn

assert np.allclose(item_emb_cs[idx_pre], item_emb[idx_pre]), "ERROR: pre-2016 modificados"

np.save(OUTPUT_EMB, item_emb_cs)

print(f"Guardado: {OUTPUT_EMB}")
print(f"  Pre-2016  ({len(idx_pre):4d}): embeddings originales")
print(f"  Post-2016 ({len(idx_post):4d}): embeddings k-NN inferidos")
print(f"  Norma media post-2016 inferida: {np.linalg.norm(y_post_knn, axis=1).mean():.4f}")
print(f"  Norma media pre-2016 real:      {np.linalg.norm(y_pre, axis=1).mean():.4f}")

Guardado: ../Data/item_embeddings_rs_coldstart.npy
  Pre-2016  (2621): embeddings originales
  Post-2016 ( 486): embeddings k-NN inferidos
  Norma media post-2016 inferida: 0.2706
  Norma media pre-2016 real:      0.7410


In [12]:
import os

print("="*70)
print("RESUMEN — Cold-Start k-NN + Developer Reputation")
print("="*70)
print(f"Metodo:         k-NN (k={K_NEIGHBORS}, similitud coseno)")
print(f"Features:       {X_all.shape[1]} total (Steam + RAWG metadata + TF-IDF)")
print(f"R2 CV:          {np.mean(cv_r2_folds):.4f} +/- {np.std(cv_r2_folds):.4f}")
print(f"Cosine sim CV:  {np.mean(cv_cosim_folds):.4f} +/- {np.std(cv_cosim_folds):.4f}")
print()
print(f"Outputs:")
size1 = os.path.getsize(OUTPUT_EMB)/1024
size2 = os.path.getsize(OUTPUT_DEVREP)/1024
print(f"  item_embeddings_rs_coldstart.npy  ({size1:.0f} KB)")
print(f"  developer_reputation.npy          ({size2:.0f} KB)")
print()
print("Proximos pasos:")
print("  Notebooks 03cs, 05cs, 08cs, 09cs, 10cs:")
print("    ITEM_EMB = '../Data/item_embeddings_rs_coldstart.npy'")
print("    + agregar developer_reputation como feature adicional")
print("="*70)

RESUMEN — Cold-Start k-NN + Developer Reputation
Metodo:         k-NN (k=10, similitud coseno)
Features:       354 total (Steam + RAWG metadata + TF-IDF)
R2 CV:          -0.0851 +/- 0.0098
Cosine sim CV:  0.0637 +/- 0.0024

Outputs:
  item_embeddings_rs_coldstart.npy  (921 KB)
  developer_reputation.npy          (15 KB)

Proximos pasos:
  Notebooks 03cs, 05cs, 08cs, 09cs, 10cs:
    ITEM_EMB = '../Data/item_embeddings_rs_coldstart.npy'
    + agregar developer_reputation como feature adicional
